<p align="center">
  <img src="https://i0.wp.com/www.tiempodecine.co/web/wp-content/uploads/2015/11/Robert-De-Niro-in-Taxi-Driver-1976.jpg?resize=750%2C375&ssl=1" style="width:100%; max-width:900px; height:180px; object-fit:cover; border-radius:10px;"/>
</p>
<div style="text-align:center;">
  <h1 style="color:#FFD700; display:inline-block; margin:0;">Optimización del Transporte en Nueva York</h1>
  <p>
    <b>Green Taxi | Machine Learning & Data Science | CRISP-DM</b><br>
    <span style="font-size:1.1em;">Análisis y predicción de tarifas y duración de viajes usando datos reales de taxis verdes de NYC.</span>
  </p>
</div>

# FASE 1: Business Understanding (CRISP-DM)

## Propósito de la Fase de Comprensión del Negocio

En esta primera fase del proceso CRISP-DM, nos enfocamos en entender los objetivos y requisitos del negocio relacionados con el proyecto de análisis de datos. Esto incluye la identificación de los problemas clave que se desean resolver, la definición de los objetivos del proyecto y la comprensión del contexto empresarial en el que se aplicarán los resultados del análisis. El objetivo es asegurar que el proyecto esté alineado con las necesidades del negocio y que los resultados sean relevantes y útiles para la toma de decisiones.

# Carga de Librerías

In [1]:
# Verifica si las librerías necesarias ya están instaladas
try:
    # Ignorar warnings
    import warnings
    warnings.filterwarnings('ignore')
    # Librerías del sistema
    import sys, subprocess
    # Importaciones base
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    # Librerías de sistema de archivos
    import os as os  
    from os import path
    import pickle as pkl
    import joblib
    import duckdb

except ImportError:

    print("Dependencias no encontradas. Instalando ahora...")
    
    # Se ejecutan los comandos de instalación
    %pip install --quiet matplotlib
    %pip install --quiet seaborn
    %pip install --quiet joblib
    %pip install --quiet scipy
    %pip install --quiet duckdb
    %pip install --quiet pyarrow
    %pip install --quiet fastparquet
    
    print("Instalación completada.")

## **Contexto del Negocio**

La Comisión de Taxis y Limusinas de Nueva York (TLC) gestiona uno de los sistemas de transporte urbano más complejos del mundo, supervisando:
- **Taxis Amarillos**: Operan en toda la ciudad, principalmente por street-hail
- **Taxis Verdes**: Operan principalmente en boroughs exteriores
- **Vehículos de Alquiler (FHV)**: Servicios previamente arreglados
- **Vehículos de Alto Volumen (HVFHV)**: Servicios de ridesharing como Uber, Lyft

La ciudad enfrenta desafíos críticos de movilidad urbana donde la **distribución desigual de la demanda** genera ineficiencias operativas, afectando tanto a proveedores de servicio como a usuarios finales.

---

## **Problema Central**

### **Desbalance Geográfico y Temporal de la Demanda**

**Evidencias Identificadas:**
- Concentración masiva de viajes en zonas específicas durante horas pico
- Subutilización de flotas en áreas periféricas y horarios valle
- Saturación en distritos financieros y comerciales vs. escasez en zonas residenciales
- Ineficiencia en la asignación de diferentes tipos de servicio (Yellow vs. Green Taxis)

**Impacto en el Negocio:**
- Tiempos de espera excesivos para pasajeros en zonas saturadas
- Bajos ingresos para conductores en zonas de baja demanda
- Congestión vehicular en áreas críticas
- Experiencia de usuario inconsistente

---

## **Objetivos de Negocio**

### **Objetivo Principal**
**Optimizar la distribución espacial y temporal de los servicios de transporte** mediante segmentación inteligente de zonas urbanas.

### **Objetivos Específicos**

1. **Segmentación por Patrones de Demanda**
   - Identificar clusters de zonas con comportamientos similares
   - Categorizar áreas según volumen, temporalidad y tipo de servicio

2. **Gestión de Capacidad Dinámica**
   - Predecir demandas pico y asignar recursos proactivamente
   - Balancear flotas entre zonas complementarias

3. **Optimización de Servicios Especializados**
   - Asignar tipos de vehículos según características zonales
   - Personalizar estrategias por segmento identificado

4. **Planificación de Infraestructura**
   - Identificar ubicaciones óptimas para estaciones de vehículos
   - Optimizar rutas y puntos de espera

---

## **Preguntas Clave de Negocio**

### **1. ¿Cómo podemos segmentar las zonas de NYC según patrones de demanda?**
- *Enfoque:* Identificar clusters naturales basados en volumen horario, tipo de servicio y características temporales
- *Métrica:* Grupos homogéneos con patrones de demanda similares
- *Impacto:* Estrategias diferenciadas por tipo de zona

### **2. ¿Qué zonas presentan mayor saturación y en qué horarios?**
- *Enfoque:* Análisis de concentración de viajes por LocationID y banda horaria
- *Métrica:* Porcentaje de viajes en horas pico vs. capacidad estimada
- *Impacto:* Redistribución prioritaria de recursos

### **3. ¿Existen diferencias significativas en el uso de Yellow vs. Green Taxis por zona?**
- *Enfoque:* Comparativa de distribución geográfica por tipo de servicio
- *Métrica:* Ratio de utilización y patrones de viaje diferenciados
- *Impacto:* Optimización de flotas especializadas

### **4. ¿Qué patrones temporales (horarios, días) definen la demanda por zona?**
- *Enfoque:* Análisis de series temporales y estacionalidad por LocationID
- *Métrica:* Patrones recurrentes y variabilidad horaria/semanal
- *Impacto:* Programación predictiva de recursos

### **5. ¿Cómo se relacionan las zonas de pickup y dropoff en términos de demanda complementaria?**
- *Enfoque:* Análisis de flujos origen-destino y correlaciones espaciales
- *Métrica:* Matrices de transición y zonas con demanda balanceada
- *Impacto:* Estrategias de reposicionamiento eficiente

---

## **Alcance del Proyecto**

### **Dataset Aplicable:**
- **Yellow Taxi**: Análisis principal por cobertura citywide
- **Green Taxi**: Complementario para boroughs exteriores
- **FHV/HVFHV**: Validación cruzada de patrones identificados

### **Criterios de Éxito:**
- Reducción del 15% en tiempos de espera en zonas saturadas
- Aumento del 10% en utilización de flotas en zonas subutilizadas
- Mejora del 20% en balance entre oferta y demanda
- Segmentación clara con al menos 5-7 clusters interpretables

### **Entregables:**
- Modelo de clustering con segmentación de zonas
- Dashboard de monitoreo de demanda en tiempo cuasi-real
- Recomendaciones estratégicas por tipo de zona
- Plan de implementación gradual por prioridad

---

# Carga Yellow Tripdata

In [2]:
# Carga del dataset
yellow_trip = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-07.parquet")
# Crear una nueva columna con el color del taxi taxi_color = 'green'
yellow_trip['taxi_color'] = 'yellow'
# Crear columna con inicial del color del taxi
yellow_trip['taxi_color_ini'] = yellow_trip['taxi_color'].map({'yellow': 'Y'})
# Agregar columna binaria para yellow and green taxis . map({'yellow': 1, 'green': 0})
yellow_trip['is_color'] = yellow_trip['taxi_color'].map({'yellow': 1, 'green': 0})

# Carga Green Tripdata

In [3]:
# Carga del dataset parquet
green_trip = pd.read_parquet("https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-07.parquet")
# Crear una nueva columna con el color del taxi taxi_color = 'green'
green_trip['taxi_color'] = 'green'
# Crear columna con inicial del color del taxi
green_trip['taxi_color_ini'] = green_trip['taxi_color'].map({'green': 'G'})
# Agregar columna binaria para yellow and green taxis . map({'yellow': 1, 'green': 0})
green_trip['is_color'] = green_trip['taxi_color'].map({'yellow': 1, 'green': 0})

---
# **Estrategia de Tratamiento de Datos**

El notebook ejecuta un proceso ETL en dos flujos paralelos para preparar los datos de Taxis y FHVHV.

Para los Taxis (Yellow/Green) , se realizó una normalización de esquema, unificando los nombres de las columnas de fechas (tpep_/lpep_ a pickup_datetime) y alineando los dataframes. Para corregir el desbalance de clases extremo (3.9M vs 48k) , se aplicó random undersampling al dataset Yellow, reduciéndolo al tamaño exacto del Green. Para los FHVHV, un pipeline basado en duckdb muestreó al 1% tres archivos Parquet mensuales (Jul a Sep). Finalmente, ambos datasets procesados (df_taxis_equilibrado y df_fhvhv_sample) se guardaron en nuevos archivos Parquet.

### Unifica nombres de columnas y asegura un esquema consistente entre taxis Yellow and Green.

In [4]:
# Normalización de columnas entre taxis amarillos y verdes
print("\nNormalizando columnas clave de datetime y esquema entre Yellow y Green...")

def alinear_esquema_taxis(df_yellow: pd.DataFrame, df_green: pd.DataFrame):
    """Unifica nombres de columnas y asegura un esquema consistente entre taxis amarillos y verdes."""
    # Renombrar columnas de fecha para que compartan identificadores consistentes
    yellow_renombrado = df_yellow.rename(
        columns={
            "tpep_pickup_datetime": "pickup_datetime",
            "tpep_dropoff_datetime": "dropoff_datetime"
        }
    ).copy()
    green_renombrado = df_green.rename(
        columns={
            "lpep_pickup_datetime": "pickup_datetime",
            "lpep_dropoff_datetime": "dropoff_datetime"
        }
    ).copy()
    
    # Conjunto unificado de columnas sin duplicados
    columnas_unificadas = sorted(set(yellow_renombrado.columns).union(set(green_renombrado.columns)))
    
    # Reindexa para garantizar que ambos DataFrames compartan el mismo orden de columnas
    yellow_alineado = yellow_renombrado.reindex(columns=columnas_unificadas)
    green_alineado = green_renombrado.reindex(columns=columnas_unificadas)
    
    # Reportar columnas exclusivas de cada dataset para trazabilidad
    columnas_solo_yellow = sorted(set(yellow_renombrado.columns) - set(green_renombrado.columns))
    columnas_solo_green = sorted(set(green_renombrado.columns) - set(yellow_renombrado.columns))
    
    if columnas_solo_yellow:
        print(f"   Columnas exclusivas Yellow ({len(columnas_solo_yellow)}): {columnas_solo_yellow}")
    if columnas_solo_green:
        print(f"   Columnas exclusivas Green ({len(columnas_solo_green)}): {columnas_solo_green}")
    
    return yellow_alineado, green_alineado, columnas_unificadas

# DataFrames alineados listos para muestreo equilibrado
yellow_trip_aligned, green_trip_aligned, columnas_unificadas = alinear_esquema_taxis(yellow_trip, green_trip)
print(f"   Columnas unificadas: {len(columnas_unificadas)}")
print(f"   Yellow columnas: {len(yellow_trip_aligned.columns)} | Green columnas: {len(green_trip_aligned.columns)}")


Normalizando columnas clave de datetime y esquema entre Yellow y Green...
   Columnas exclusivas Yellow (1): ['Airport_fee']
   Columnas exclusivas Green (2): ['ehail_fee', 'trip_type']
   Columnas unificadas: 25
   Yellow columnas: 25 | Green columnas: 25


### Balanceo de la carga de los Datos para Taxis NY Yellow and Green

In [5]:
# BALANCED SAMPLING USING ORIGINAL GREEN SIZE
print("\n" + "=" * 70)
print("BALANCED SAMPLING: YELLOW = ORIGINAL GREEN SIZE")
print("=" * 70)

def create_yellow_sample_equal_original_green(df_yellow, df_green):
    """
    Create a sample where Yellow has exactly the same size as ORIGINAL Green.
    Green remains complete without modifications.
    Assumes both DataFrames comparten el mismo esquema y columnas únicas.
    
    Args:
        df_yellow: Yellow taxi DataFrame already aligned (pickup/dropoff unificados)
        df_green: Green taxi DataFrame already aligned (pickup/dropoff unificados)
    
    Returns:
        tuple: (yellow_sample, green_complete) where both have the same size
    """
    
    df_yellow = df_yellow.copy()
    df_green = df_green.copy()
    
    print("CREATING BALANCED DATASET WITH ORIGINAL GREEN SIZE...")
    
    # Original statistics
    total_yellow = len(df_yellow)
    total_green = len(df_green)
    
    print("ORIGINAL ANALYSIS:")
    print(f"   Original Yellow: {total_yellow:,} records")
    print(f"   Original Green:  {total_green:,} records")
    print(f"   Difference:      {total_yellow - total_green:,} records")
    
    # Target is to make Yellow the same size as original Green
    target_size = total_green
    
    print("BALANCING TARGET:")
    print(f"   Yellow target: {target_size:,} records (= Original Green)")
    print(f"   Green target:  {total_green:,} records (complete, unchanged)")
    print(f"   Final total:   {target_size * 2:,} records")
    
    # Create samples
    print("PROCESSING DATASETS...")
    
    # Yellow: subsample to Green size
    if total_yellow >= target_size:
        yellow_sample = df_yellow.sample(n=target_size, random_state=42)
        print(f"   SUCCESS Yellow: subsampled from {total_yellow:,} to {target_size:,}")
    else:
        # Unlikely case where Green is larger than Yellow
        yellow_sample = df_yellow.copy()
        print(f"   WARNING Yellow: full dataset kept ({len(yellow_sample):,} records - smaller than Green)")
    
    # Green: keep complete
    green_complete = df_green.copy()
    print(f"   SUCCESS Green: full dataset kept ({len(green_complete):,} records)")
    
    # Verify final result
    total_final = len(yellow_sample) + len(green_complete)
    yellow_percentage = len(yellow_sample) / total_final * 100
    green_percentage = len(green_complete) / total_final * 100
    
    print("FINAL RESULT:")
    print(f"   Final Yellow: {len(yellow_sample):,} records ({yellow_percentage:.2f}%)")
    print(f"   Final Green:  {len(green_complete):,} records ({green_percentage:.2f}%)")
    print(f"   Final total:  {total_final:,} records")
    
    # Check if balanced
    size_difference = abs(len(yellow_sample) - len(green_complete))
    is_balanced = size_difference == 0
    
    print(f"   Size difference: {size_difference:,} records")
    print(f"   Perfectly balanced: {is_balanced}")
    
    if is_balanced:
        print(f"   SUCCESS: Both datasets have exactly {len(green_complete):,} records")
    
    return yellow_sample, green_complete

def show_complete_final_comparison(yellow_orig, green_orig, yellow_eq, green_eq):
    """Show complete comparison of all methods"""
    print("\n" + "=" * 90)
    print("COMPLETE COMPARISON: ORIGINAL vs BALANCED (YELLOW = ORIGINAL GREEN)")
    print("=" * 90)
    
    # Original statistics
    total_orig = len(yellow_orig) + len(green_orig)
    yellow_orig_pct = len(yellow_orig) / total_orig * 100
    green_orig_pct = len(green_orig) / total_orig * 100
    
    # Balanced statistics
    total_eq = len(yellow_eq) + len(green_eq)
    yellow_eq_pct = len(yellow_eq) / total_eq * 100
    green_eq_pct = len(green_eq) / total_eq * 100
    
    # Comparative table
    print(f"{'Method':<20} {'Total':<12} {'Yellow':<12} {'Green':<12} {'Yellow %':<10} {'Green %':<10}")
    print("-" * 90)
    print(f"{'Original':<20} {total_orig:<12,} {len(yellow_orig):<12,} {len(green_orig):<12,} {yellow_orig_pct:<10.2f} {green_orig_pct:<10.2f}")
    print(f"{'Balanced':<20} {total_eq:<12,} {len(yellow_eq):<12,} {len(green_eq):<12,} {yellow_eq_pct:<10.2f} {green_eq_pct:<10.2f}")
    
    # Reduction calculations
    yellow_reduction = len(yellow_orig) - len(yellow_eq)
    total_reduction = total_orig - total_eq
    
    print(f"\nBALANCING STATISTICS:")
    print(f"   • Yellow reduced by: {yellow_reduction:,} records")
    print(f"   • Green maintained:  {len(green_eq):,} records (no changes)")
    print(f"   • Total reduced by:  {total_reduction:,} records")
    print(f"   • Final proportion:  50.00% Yellow / 50.00% Green")
    
    print(f"\nBALANCING ADVANTAGES:")
    print(f"   SUCCESS Eliminates class imbalance (98.78% vs 1.22%)")
    print(f"   SUCCESS Maintains all Green data (no loss)")
    print(f"   SUCCESS Perfect for Machine Learning algorithms")
    print(f"   SUCCESS Ideal 50/50 proportion for classification")

# Execute sampling with original Green size
print("EXECUTING SAMPLING: YELLOW = ORIGINAL GREEN SIZE...")
yellow_balanced_real, green_complete = create_yellow_sample_equal_original_green(
    yellow_trip_aligned, 
    green_trip_aligned
)

# Show complete comparison
show_complete_final_comparison(
    yellow_trip_aligned, green_trip_aligned,           # Originals alineados
    yellow_balanced_real, green_complete  # Balanced with original Green size
)


BALANCED SAMPLING: YELLOW = ORIGINAL GREEN SIZE
EXECUTING SAMPLING: YELLOW = ORIGINAL GREEN SIZE...
CREATING BALANCED DATASET WITH ORIGINAL GREEN SIZE...
ORIGINAL ANALYSIS:
   Original Yellow: 3,898,963 records
   Original Green:  48,205 records
   Difference:      3,850,758 records
BALANCING TARGET:
   Yellow target: 48,205 records (= Original Green)
   Green target:  48,205 records (complete, unchanged)
   Final total:   96,410 records
PROCESSING DATASETS...
   SUCCESS Yellow: subsampled from 3,898,963 to 48,205
   SUCCESS Green: full dataset kept (48,205 records)
FINAL RESULT:
   Final Yellow: 48,205 records (50.00%)
   Final Green:  48,205 records (50.00%)
   Final total:  96,410 records
   Size difference: 0 records
   Perfectly balanced: True
   SUCCESS: Both datasets have exactly 48,205 records

COMPLETE COMPARISON: ORIGINAL vs BALANCED (YELLOW = ORIGINAL GREEN)
Method               Total        Yellow       Green        Yellow %   Green %   
-----------------------------------

# Carga del Dataset Completo Trip Data  FHVHV 06-2025

In [6]:
# --- 1. CONFIGURACIÓN DEL PIPELINE  ---

# Llaves corregidas para que coincidan con los datos (Jul, Ago, Sep)
FHVHV_URLS = {
    "fhvhv_julio_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-07.parquet",
    "fhvhv_agosto_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-08.parquet",
    "fhvhv_septiembre_2025": "https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-09.parquet"
}
# 
MODO_DESARROLLO = True  # Cambiar a False para producción

#### Carga y concatena múltiples archivos Parquet

In [7]:
# --- 2. LA FUNCIÓN DEL PIPELINE CON BALANCEADO EQUILIBRADO ---
def crear_dataset_final(fuentes_url: dict, modo_dev: bool = True, colores_fuentes: dict = None, equilibrar_fuentes: bool = True):
    """
    Carga y concatena múltiples archivos Parquet desde URLs o DataFrames con balanceado equilibrado.
    
    - Si modo_dev=True: Toma una muestra rápida del 1% de cada archivo.
    - Si modo_dev=False: Carga el 100% de cada archivo.
    - Si equilibrar_fuentes=True: Balancea el número de registros entre fuentes.
    - Si colores_fuentes está definido, añade la columna 'taxi_color' usando ese mapping.
    """
    lista_dataframes = []
    tamaños_originales = {}
    
    if modo_dev:
        print(f"--- MODO DESARROLLO (Muestra: 1%) ---")
        sample_fraction = 0.01  # 1%
    else:
        print(f"--- MODO PRODUCCIÓN (Datos: 100%) ---")
        sample_fraction = 1.0  # 100%
    
    if equilibrar_fuentes:
        print(f"MODO EQUILIBRADO ACTIVADO - Balanceando fuentes de datos")

    con = duckdb.connect(database=':memory:')
    
    # FASE 1: Cargar datos iniciales y obtener tamaños
    for nombre_fuente, fuente in fuentes_url.items():
        print(f"Procesando: {nombre_fuente}...")
        
        try:
            # Si es un DataFrame de pandas, lo usamos directamente
            if isinstance(fuente, pd.DataFrame):
                if modo_dev:
                    df_temp = fuente.sample(frac=sample_fraction, random_state=42)
                else:
                    df_temp = fuente.copy()
                print(f"-> Cargado desde DataFrame: {len(df_temp)} filas")
            
            # Si es una URL, usamos DuckDB para cargarlo
            elif isinstance(fuente, str):
                if modo_dev:
                    query = f"""
                    SELECT *
                    FROM '{fuente}'
                    USING SAMPLE {int(sample_fraction * 100)}%
                    """
                else:
                    query = f"""
                    SELECT *
                    FROM '{fuente}'
                    """
                
                # Ejecutar la consulta y convertir a DataFrame
                df_temp = con.execute(query).df()
                print(f"-> Cargado desde URL: {len(df_temp)} filas")
            
            else:
                print(f"-> ERROR: Tipo de fuente no soportado para {nombre_fuente}")
                continue
            
            # Añadir metadatos de procedencia
            if not df_temp.empty:
                df_temp = df_temp.copy()
                df_temp['fuente_origen'] = nombre_fuente
                lista_dataframes.append(df_temp)
                tamaños_originales[nombre_fuente] = len(df_temp)
            else:
                print(f"-> Advertencia: {nombre_fuente} está vacío")
                
        except Exception as e:
            print(f"-> ERROR procesando {nombre_fuente}: {e}")
            continue
    
    con.close()

    print(f"\nConcatenando DataFrames equilibrados...")
    df_final = pd.concat(lista_dataframes, axis=0, ignore_index=True)
    
    # Mezclar el DataFrame final para distribuir las fuentes
    df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"¡Proceso completado! Total de filas: {len(df_final):,}")
    
    # Mostrar distribución final por fuente
    if 'fuente_origen' in df_final.columns:
        print(f"\nDistribución final por fuente:")
        distribucion_final = df_final['fuente_origen'].value_counts()
        for fuente, cantidad in distribucion_final.items():
            porcentaje = (cantidad / len(df_final)) * 100
            print(f"   • {fuente}: {cantidad:,} registros ({porcentaje:.1f}%)")
    
    return df_final

# Creación de Muestra 1% del Dataset FHVHV - Taxi Verde & Taxi Amarillo

In [8]:
# --- 3. EJECUCIÓN DEL PIPELINE ---

print("=== Iniciando Pipeline SOLO para FHVHV (Uber/Lyft) ===")
df_fhvhv_sample = crear_dataset_final(FHVHV_URLS, modo_dev=MODO_DESARROLLO)

print("\n" + "="*60 + "\n")

print("=== Dataset TAXIS (Yellow/Green) - YA EQUILIBRADO ===")
print("Los taxis no pasan por create_dataset_final, se usan directamente:")
print(f"• Yellow equilibrado: {len(yellow_balanced_real):,} registros")
print(f"• Green completo:     {len(green_complete):,} registros")

# Concatenar directamente los taxis equilibrados
df_taxis_equilibrado = pd.concat([yellow_balanced_real, green_complete], ignore_index=True)
df_taxis_equilibrado = df_taxis_equilibrado.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"• Taxis equilibrados totales: {len(df_taxis_equilibrado):,} registros")

print("\n" + "="*60 + "\n")
print("Procesamiento finalizado:")
print(f"1. df_fhvhv_sample: {len(df_fhvhv_sample):,} filas (pipeline)")
print(f"2. df_taxis_equilibrado: {len(df_taxis_equilibrado):,} filas (equilibrado directo)")

=== Iniciando Pipeline SOLO para FHVHV (Uber/Lyft) ===
--- MODO DESARROLLO (Muestra: 1%) ---
MODO EQUILIBRADO ACTIVADO - Balanceando fuentes de datos
Procesando: fhvhv_julio_2025...
-> Cargado desde URL: 198656 filas
Procesando: fhvhv_agosto_2025...
-> Cargado desde URL: 208896 filas
Procesando: fhvhv_septiembre_2025...
-> Cargado desde URL: 157696 filas

Concatenando DataFrames equilibrados...
¡Proceso completado! Total de filas: 565,248

Distribución final por fuente:
   • fhvhv_agosto_2025: 208,896 registros (37.0%)
   • fhvhv_julio_2025: 198,656 registros (35.1%)
   • fhvhv_septiembre_2025: 157,696 registros (27.9%)


=== Dataset TAXIS (Yellow/Green) - YA EQUILIBRADO ===
Los taxis no pasan por create_dataset_final, se usan directamente:
• Yellow equilibrado: 48,205 registros
• Green completo:     48,205 registros
• Taxis equilibrados totales: 96,410 registros


Procesamiento finalizado:
1. df_fhvhv_sample: 565,248 filas (pipeline)
2. df_taxis_equilibrado: 96,410 filas (equilibrado 

# Carga del Dataset FHVHV (Uber/Lyft)  & TAXIS (Yellow/Green)

### FHVHV (Uber/Lyft)

In [9]:
# FHVHV (Uber/Lyft)
print("Guardando muestra de FHVHV (Uber/Lyft)...")
archivo_fhvhv = 'fhvhv_trimestral.parquet'

try:
    df_fhvhv_sample.to_parquet(archivo_fhvhv, index=False)
    print(f"-> ¡Éxito! Guardado como '{archivo_fhvhv}'")
except Exception as e:
    print(f"-> ERROR al guardar FHVHV: {e}")

Guardando muestra de FHVHV (Uber/Lyft)...
-> ¡Éxito! Guardado como 'fhvhv_trimestral.parquet'


### TAXIS (Yellow/Green)


In [10]:
# TAXIS (Yellow/Green) - Dataset Equilibrado Directo
print("\nGuardando dataset EQUILIBRADO de TAXIS (Yellow/Green)...")
archivo_taxis_equilibrado = 'taxis_equilibrado_directo.parquet'

try:
    df_taxis_equilibrado.to_parquet(archivo_taxis_equilibrado, index=False)
    print(f"-> ¡Éxito! Guardado como '{archivo_taxis_equilibrado}'")
    print(f"-> Total registros: {len(df_taxis_equilibrado):,}")
    
    # Verificar distribución
    if 'taxi_color' in df_taxis_equilibrado.columns:
        distribucion = df_taxis_equilibrado['taxi_color'].value_counts()
        print(f"-> Distribución por color:")
        for color, cantidad in distribucion.items():
            porcentaje = (cantidad / len(df_taxis_equilibrado)) * 100
            print(f"   • {color.title()}: {cantidad:,} ({porcentaje:.1f}%)")
            
except Exception as e:
    print(f"-> ERROR al guardar Taxis: {e}")


Guardando dataset EQUILIBRADO de TAXIS (Yellow/Green)...
-> ¡Éxito! Guardado como 'taxis_equilibrado_directo.parquet'
-> Total registros: 96,410
-> Distribución por color:
   • Green: 48,205 (50.0%)
   • Yellow: 48,205 (50.0%)


### Carga de Datos Procesados

In [11]:
# Cargar datasets procesados
taxis_equilibrado_final = pd.read_parquet('taxis_equilibrado_directo.parquet')
fhvhv_sample = pd.read_parquet('fhvhv_trimestral.parquet')

print("Datasets cargados:")
print(f"• Taxis equilibrados: {len(taxis_equilibrado_final):,} registros")
print(f"• FHVHV sample: {len(fhvhv_sample):,} registros")

Datasets cargados:
• Taxis equilibrados: 96,410 registros
• FHVHV sample: 565,248 registros
